In [30]:
import pandas as pd
import numpy as np
import datetime
import os
from pandas.api.types import is_integer_dtype
from pandas.api.types import is_numeric_dtype

In [3]:
raw_path = '/trials/vaccine/p807/s001/qdata/LabData/BCR_sequencing_pass-through/uploaded_by_lab/McElrath/20260417-02/2026-04-17_HVTN807_BCS2098-BCS2103_AIRR_filtered.tsv'
raw = pd.read_csv(raw_path, sep="\t")

/tmp/ipykernel_3354565/1871316888.py:2: DtypeWarning: Columns (11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv(raw_path, sep="\t")


In [4]:
savedir = '/networks/vtn/lab/SDMC_labscience/studies/HVTN/HVTN807/assays/BCR_sequencing/misc_files/data_processing/'
os.listdir(savedir)

['HVTN807_BCR_sample_summary_2026-04-17.xlsx',
 'HVTN807_BCR_sequencing_regenerated_AIRR_wide_processed_2026-04-30.txt',
 'HVTN807_BCR_updated_sample_summary_2026-04-17.xlsx',
 'check_dropped_cell_ids.xlsx',
 '2026-04-28_HVTN807_BCS2107-BCS2108_AIRR_filtered_sample_summary_2026-04-29.xlsx',
 'HVTN807_BCR_sequencing_regenerated_AIRR_long_pivot_summary_2026-04-29.xlsx',
 '~$2026-04-28_HVTN807_BCS2107-BCS2108_AIRR_filtered_sample_summary_2026-04-29.xlsx',
 'archive',
 'HVTN807_BCR_sequencing_regenerated_AIRR_long_processed_2026-04-30.txt']

In [5]:
long = pd.read_csv(
    savedir + "HVTN807_BCR_sequencing_regenerated_AIRR_long_processed_2026-04-30.txt",
    sep='\t'
)
wide = pd.read_csv(
    savedir + 'HVTN807_BCR_sequencing_regenerated_AIRR_wide_processed_2026-04-30.txt',
    sep='\t'
)

/tmp/ipykernel_3354565/573457033.py:1: DtypeWarning: Columns (146,160) have mixed types. Specify dtype option on import or set low_memory=False.
  long = pd.read_csv(
/tmp/ipykernel_3354565/573457033.py:5: DtypeWarning: Columns (42) have mixed types. Specify dtype option on import or set low_memory=False.
  wide = pd.read_csv(


In [6]:
raw.Global_Spec_ID = raw.Global_Spec_ID.str.replace("\t","")

In [10]:
# cell is is a unique record ID
assert raw.cell_id.nunique() == len(raw)

In [9]:
# same guspecs in all files
assert set(raw.Global_Spec_ID).symmetric_difference(long.guspec_core) == set()

In [12]:
# all cell_ids in outputs are in raw
assert set(long.cell_id).difference(raw.cell_id) == set()

In [13]:
# there are 10995 cell ids in the raw that arent in the retagged files
len(set(raw.cell_id).difference(wide.cell_id))

10995

In [15]:
# cell id is a unique id in the wide
assert len(wide) == wide.cell_id.nunique()

In [16]:
# cell id + chain are a unique id in the long
assert len(long) == long.cell_id.nunique()*2
assert len(long[['cell_id','chain']].drop_duplicates()) == len(long)

In [ ]:
## CHECKS FROM SARA'S SPEC

In [17]:
assert set(long.chain.unique()).difference(['heavy','light']) == set()

In [19]:
assert set(long.locus.unique()).difference(['IGH', 'IGK', 'IGL']) == set()

In [21]:
assert set(long.stop_codon.unique()).difference(['T','F']) == set()

In [22]:
assert set(long.vj_in_frame.unique()).difference(['T','F']) == set()

In [24]:
assert set(long.v_frameshift.unique()).difference(['T','F']) == set()

In [26]:
assert set(long.productive.unique()).difference(['T','F']) == set()

In [27]:
assert set(long.rev_comp.unique()).difference(['T','F']) == set()

In [28]:
assert set(long.complete_vdj.unique()).difference(['T','F']) == set()

In [29]:
assert set(long.d_frame.dropna().unique().astype(int)).difference([1,2,3]) == set()

In [31]:
for col in [
    'v_alignment_start',
    'v_alignment_end',
    'j_alignment_start',
    'j_alignment_end',
    # 'c_alignment_start', #these have nulls? we've asked ju via email, 4/30
    # 'c_alignment_end', #these have nulls? we've asked ju via email, 4/30
]:
    assert is_integer_dtype(long[col])
    assert long[col].min() > 0

In [32]:
for col in [
    'd_alignment_start',
    'd_alignment_end',
]:
    assert is_integer_dtype(long[col].dropna().astype(int))
    assert long[col].dropna().min() > 0

In [33]:
assert is_integer_dtype(long.junction_length.dropna().astype(int))

In [34]:
assert is_integer_dtype(long['junction_aa_length'].dropna().astype(int))

In [36]:
for col in ['v_score','d_score','j_score','c_score']:
    assert is_numeric_dtype(long[col])
    assert long[col].min() >= 0

In [37]:
for col in ['v_support','d_support','j_support','c_support']:
    assert is_numeric_dtype(long[col])
    assert long[col].min() >= 0

In [38]:
for col in ['v_identity','d_identity','j_identity','c_identity']:
    assert is_numeric_dtype(long[col])
    assert long[col].min() >= 0
    assert long[col].min() <= 100

In [39]:
for col in [
    'v_sequence_start','v_sequence_end','v_germline_start','v_germline_end',
    'j_sequence_start',
    'j_sequence_end',
    'j_germline_start',
    'j_germline_end',
    # 'c_sequence_start',# has nulls
    # 'c_sequence_end',
    # 'c_germline_start',
    # 'c_germline_end',
]:
    assert is_integer_dtype(long[col].astype(int))
    assert long[col].min() > 0

In [40]:
for col in [
    'c_sequence_start',
    'c_sequence_end',
    'c_germline_start',
    'c_germline_end',
]:
    print(long[col].isna().sum())
    assert is_integer_dtype(long[col].dropna().astype(int))
    assert long[col].min() > 0

1142
1142
1142
1142


In [41]:
for col in [
    'd_sequence_start',
    'd_sequence_end',
    'd_germline_start',
    'd_germline_end',
]:
    assert is_integer_dtype(long[col].dropna().astype(int))
    assert long[col].min() > 0

In [42]:
for col in [
    'fwr1_start',
    'fwr1_end',
    'cdr1_start',
    'cdr1_end',
    'fwr2_start',
    'fwr2_end',
    'cdr2_start',
    'cdr2_end',
    'fwr3_start',
    'fwr3_end',
    'fwr4_start',
    'fwr4_end',
    'cdr3_start',
    'cdr3_end',
]:
    try:
        assert is_integer_dtype(long[col].astype(int))
        assert long[col].min() > 0
    except:
        print(f"{col} - {long[col].isna().sum()}")
        # print(outputs[col].isna().sum())
        assert is_integer_dtype(long[col].dropna().astype(int))
        assert long[col].dropna().min() > 0

cdr1_start - 4
cdr1_end - 4
fwr2_start - 18
fwr2_end - 18
cdr2_start - 35
cdr2_end - 35
fwr3_start - 41
fwr3_end - 41
fwr4_start - 134
fwr4_end - 134
cdr3_start - 134
cdr3_end - 134


In [43]:
for col in ['np1_length','np2_length']:
    assert is_integer_dtype(long[col].dropna().astype(int))
    assert long[col].dropna().min() >= 0

In [45]:
[i for i in long.columns if '_candidate' in i]

['is_vrc01_candidate',
 'is_ioma_candidate',
 'is_vh146_candidate',
 'v2apex_candidate_type',
 'is_v2apex_candidate',
 'is_ch103_candidate',
 'is_hj16_candidate',
 'is_vrc13_candidate',
 'is_vrc16_candidate',
 'bg18_candidate_type',
 'is_bg18_candidate']

In [46]:
for i in [
    'is_vrc01_candidate',
    'is_ioma_candidate',
    'is_vh146_candidate',
    'is_v2apex_candidate',
    'is_ch103_candidate',
    'is_hj16_candidate',
    'is_vrc13_candidate',
    'is_vrc16_candidate',
    'is_bg18_candidate'
         ]:
    assert set(long[i]).difference([True, False]) == set()

In [47]:
# these match what ju indicated over email. 1.0 seems it ought to be '1'; we've emailed him to ask, 4/30
for col in ['v2apex_candidate_type','bg18_candidate_type',]:
    print(long[col].unique())

[nan '1' '2-YYD' '2-DDY' 1.0]
[nan 'I-1' 'II' 'I-2']


In [48]:
assert is_integer_dtype(long.clone_size.astype(int))

In [49]:
assert len(long.loc[(long.const.notna()) & (long.const.str[:3] != long.locus),['const','locus']]) == 0

In [50]:
for col in ['np1_length','np2_length']:
    assert is_integer_dtype(long[col].dropna().astype(int))
    assert long[col].dropna().min() >= 0

In [52]:
assert (100 - long.v_identity - long.v_mutation_pct).abs().max() < 1e-14

In [53]:
assert is_integer_dtype(long.cdr3_aa_length)
assert long.cdr3_aa_length.min() >= 0

In [54]:
# these look acceptable
long.epitope_specificity.unique()

array(['Antigen-negative', 'Epitope-specific', 'Non-epitope-specific'],
      dtype=object)

In [55]:
# these look acceptable
long.bnab_class.unique()

array([nan, 'VRC01-class'], dtype=object)

In [56]:
d_cols = [
    'd_call',
    'd_alignment_start',
    'd_alignment_end',
    'd_sequence_alignment_aa',
    'd_germline_alignment',
    'd_germline_alignment_aa',
    'd_cigar',
    'd_support',
    'd_identity',
    'd_sequence_start',
    'd_sequence_end',
    'd_germline_start',
    'd_germline_end',
]
for d in d_cols:
    light_nulls = long.loc[long.chain=='light',d_cols[0]].isna().sum()
    heavy_nulls = long.loc[long.chain=='heavy',d_cols[0]].isna().sum() 
    try:
        assert light_nulls == (long.chain=='light').sum()
    except:
        print(f"{d} non null for {light_nulls} light rows")
    try:
        assert heavy_nulls == 0
    except:
        print(f"{d} null for {heavy_nulls} heavy rows")


d_call null for 2869 heavy rows
d_alignment_start null for 2869 heavy rows
d_alignment_end null for 2869 heavy rows
d_sequence_alignment_aa null for 2869 heavy rows
d_germline_alignment null for 2869 heavy rows
d_germline_alignment_aa null for 2869 heavy rows
d_cigar null for 2869 heavy rows
d_support null for 2869 heavy rows
d_identity null for 2869 heavy rows
d_sequence_start null for 2869 heavy rows
d_sequence_end null for 2869 heavy rows
d_germline_start null for 2869 heavy rows
d_germline_end null for 2869 heavy rows


In [57]:
from Bio.Data import IUPACData

In [58]:
valid_aa = set(IUPACData.protein_letters)

In [59]:
nucleotide_cols = [
    'sequence',
    'sequence_alignment',
    'germline_alignment',
    'v_sequence_alignment',
    'v_sequence_alignment',
    'v_germline_alignment',
    'd_sequence_alignment',
    'd_germline_alignment',
    'j_sequence_alignment',
    'j_germline_alignment',
    'c_sequence_alignment',
    'c_germline_alignment',
    'fwr1',
    'cdr1',
    'fwr2',
    'cdr2',
    'fwr3',
    'fwr4',
    'cdr3',
    'np1',
    'np2',
    'junction',
]

In [60]:
for c in nucleotide_cols:
    nulls = long[c].isna().sum()
    if nulls > 0:
        print(f"{c}, nulls: {nulls}")
    leftovers = set("".join(long[c].dropna().astype(str))).difference({'-', 'A', 'C', 'G', 'T','N'})
    try:
        assert leftovers == set()
    except:
        print(f"{c}, {leftovers}")

d_sequence_alignment, nulls: 126724
d_germline_alignment, nulls: 126724
c_sequence_alignment, nulls: 1142
c_germline_alignment, nulls: 1142
cdr1, nulls: 4
fwr2, nulls: 18
cdr2, nulls: 35
fwr3, nulls: 41
fwr4, nulls: 134
cdr3, nulls: 134
np1, nulls: 57932
np1, {'(', ')'}
np2, nulls: 137215
np2, {'(', ')'}
junction, nulls: 134


In [61]:
long.loc[(long.np1.notna()) & (long.np1.str.contains("\)")),'np1']

66866         (AGG)
125227          (G)
134860     (CTACGC)
146702          (C)
148378    (TGGGTCC)
230267        (CAC)
234476       (CACC)
Name: np1, dtype: object

In [62]:
long.loc[(long.np2.notna()) & (long.np2.str.contains("\)")),'np2']

32888    (C)
Name: np2, dtype: object

In [63]:
aa_codes = [
    'sequence_aa',
    'sequence_alignment_aa',
    'germline_alignment_aa',
    'v_sequence_alignment_aa',
    'v_germline_alignment_aa',
    'd_sequence_alignment_aa',
    'd_germline_alignment_aa',
    'j_sequence_alignment_aa',
    'j_germline_alignment_aa',
    'c_sequence_alignment_aa',
    'c_germline_alignment_aa',
    'fwr1_aa',
    'cdr1_aa',
    'fwr2_aa',
    'cdr2_aa',
    'fwr3_aa',
    'fwr4_aa',
    'cdr3_aa',
    'junction_aa',
]

for c in aa_codes:
    nulls = long[c].isna().sum()
    if nulls > 0:
        print(f"{c}, nulls: {nulls}")
    leftovers = set("".join(long[c].dropna().astype(str))).difference(valid_aa)
    try:
        assert leftovers == set()
    except:
        print(f"{c}, {leftovers}")

sequence_aa, {'*'}
sequence_alignment_aa, {'-'}
germline_alignment_aa, {'X', '*', '-'}
v_sequence_alignment_aa, {'-'}
v_germline_alignment_aa, {'*', '-'}
d_sequence_alignment_aa, nulls: 126724
d_germline_alignment_aa, nulls: 126724
d_germline_alignment_aa, {'*', '-'}
j_sequence_alignment_aa, {'-'}
j_germline_alignment_aa, {'-'}
c_sequence_alignment_aa, nulls: 1142
c_sequence_alignment_aa, {'*'}
c_germline_alignment_aa, nulls: 1142
c_germline_alignment_aa, {'*'}
cdr1_aa, nulls: 4
fwr2_aa, nulls: 19
cdr2_aa, nulls: 38
fwr3_aa, nulls: 41
fwr4_aa, nulls: 134
cdr3_aa, nulls: 134
junction_aa, nulls: 134


In [64]:
cigar_chars = {'M', 'I', 'D', 'N', 'S', 'H', 'P', '=', 'X','0', '1', '2', '3', '4', '5', '6', '7', '8', '9'}
for c in ['v_cigar','d_cigar','j_cigar','c_cigar']:
    nulls = long[c].isna().sum()
    if nulls > 0:
        print(f"{c}, nulls: {nulls}")
    remainder = set("".join(long[c].dropna().astype(str))).difference(cigar_chars)
    try:
        assert remainder == set()
    except:
        print(f"{c}, {remainder}")

d_cigar, nulls: 126724
c_cigar, nulls: 1142


In [67]:
probs = {}
for col in ['v','d','j','c']:
    pattern0 = f"^IG[HKL]{col.upper()}" + "\d{1,2}[A-Z]{0,2}-\d{1,2}\*[i]{0,1}\d{1,2}"
    pattern0a = f"^IG[HKL]{col.upper()}" + "\d{1,2}[A-Z]{0,2}-NL\d{1,2}\*[i]{0,1}\d{1,2}"
    
    to_check = []
    check_v  = long[f"{col}_call"].str.split(",", expand=True)
    for i in range(check_v.shape[1]):
        vv = check_v[i].dropna()
        check_v0 = vv.str.match(pattern0)
        check_v0a = vv.str.match(pattern0a)
        to_check += vv.loc[~check_v0 & ~check_v0a].drop_duplicates().tolist() #+ vv.loc[vv.str.contains("_")].drop_duplicates().tolist()
    probs[col] = list(np.unique(to_check))

In [68]:
[len(i) for i in probs.values()]

[26, 0, 22, 40]

In [69]:
for k in probs.keys():
    upper = 40
    n = len(probs[k])
    probs[k] += [""]*(upper-n)

In [71]:
long.j_call.unique()

array(['IGHJ4*02', 'IGKJ1*01', 'IGKJ5*01', 'IGKJ4*01', 'IGHJ5*02',
       'IGLJ2*01', 'IGHJ1*01', 'IGHJ4*02,IGHJ5*02', 'IGKJ2*01',
       'IGKJ3*01', 'IGKJ2*01,IGKJ2*04', 'IGKJ2*01,IGKJ2*03,IGKJ2*04',
       'IGKJ2*03', 'IGKJ2*04', 'IGLJ7*01', 'IGLJ3*02', 'IGLJ1*01',
       'IGLJ2*01,IGLJ3*02', 'IGHJ1*01,IGHJ4*02,IGHJ5*02', 'IGHJ3*02',
       'IGKJ2*01,IGKJ2*03', 'IGHJ6*02', 'IGHJ6*03', 'IGHJ2*01',
       'IGHJ1*01,IGHJ4*02', 'IGHJ1*01,IGHJ5*02', 'IGKJ2*03,IGKJ2*04',
       'IGHJ6*02,IGHJ6*03', 'IGLJ6*01', 'IGKJ1*01,IGKJ2*01,IGKJ3*01',
       'IGLJ2*01,IGLJ3*02,IGLJ6*01', 'IGKJ2*03,IGKJ2*04,IGKJ4*01',
       'IGKJ1*01,IGKJ2*01', 'IGLJ2*01,IGLJ3*02,IGLJ7*01',
       'IGLJ3*02,IGLJ5*02', 'IGKJ2*01,IGKJ3*01', 'IGKJ3*01,IGKJ4*01',
       'IGLJ7*02', 'IGKJ1*01,IGKJ4*01', 'IGKJ1*01,IGKJ2*04',
       'IGLJ3*02,IGLJ7*01', 'IGHJ2*01,IGHJ4*02,IGHJ5*02',
       'IGLJ3*02,IGLJ6*01', 'IGHJ2*01,IGHJ6*02', 'IGHJ2*01,IGHJ4*02',
       'IGLJ3*02,IGLJ5*01', 'IGHJ4*02,IGHJ6*02',
       'IGLJ1*01,IGLJ2*01

In [70]:
pd.DataFrame(probs)

,v,d,j,c
0,IGHV1-69-2*01,,IGHJ1*01,IGHA1*01a
1,IGHV2-70D*04,,IGHJ2*01,IGHA1*01b
2,IGHV2-70D*14,,IGHJ3*02,IGHA2*01
3,IGHV3-30-3*01,,IGHJ4*02,IGHA2*02a
4,IGHV3-43D*03,,IGHJ5*02,IGHD*01a
5,IGHV3-43D*04,,IGHJ6*02,IGHD*01b
6,IGHV3-43D*05,,IGHJ6*03,IGHE*02
7,IGHV3-64D*06,,IGKJ1*01,IGHE*04a
8,IGHV3-64D*08,,IGKJ2*01,IGHG1*01
9,IGHV3-64D*09,,IGKJ2*03,IGHG1*02
